In [1]:
import dask
import flox.xarray
import logging
import os
import psutil
import sys
import warnings
import numpy as np
import pandas as pd
import xarray as xr
from omegaconf import OmegaConf
from prototypes.ocn_only.emulator import OcnTrainer

from dask.distributed import LocalCluster, Client

In [2]:
# Meta
prototype = "R4"
num_missing_samples = 0
norm = True
lat_chunk = 8
lon_chunk = 16

In [ ]:
# Dask client
cluster = LocalCluster(n_workers=64,
                       threads_per_worker=2, 
                       memory_limit='8GB')
client = Client(cluster)
client

In [4]:
# Emulator 
config_trainer_path = f"../ocn_only/{prototype}/config.yaml"
trainer_config = OmegaConf.load(config_trainer_path)
emulator = OcnTrainer(config=trainer_config)

In [5]:
# Data
fname = "targets.zarr"
chunks = {"lat":lat_chunk, "lon":lon_chunk, "sample":-1, "channels":1}
ds_targets = xr.open_zarr(os.path.join(emulator.local_store_path,
    "training", fname), chunks="auto").astype("float32")
ds_targets = ds_targets.chunk(chunks).persist()

/global/homes/n/nagarwal/.conda/envs/graphufs-mpi/lib/python3.11/site-packages/distributed/client.py:3362: UserWarning: Sending large graph of size 18.64 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [6]:
# Build datetime coordinate for sample
target_lead_time = int(emulator.target_lead_time[:-1])
delta_t_data = int(emulator.delta_t_data[:-1])
start = pd.Timestamp(emulator.training_dates[0])
end = pd.Timestamp(emulator.training_dates[-1])
targets_time = pd.date_range(
    start = start + pd.Timedelta(target_lead_time, "h"),
    end = end - int(num_missing_samples)*pd.Timedelta(delta_t_data, "h"),
    freq=emulator.delta_t_data,
    inclusive="both"
)
ds_targets = ds_targets.assign_coords(datetime=("sample", targets_time.values))

In [7]:
# subsample along time
step = int(target_lead_time/delta_t_data)
ds_targets = ds_targets.isel(sample=slice(None, 1000, step)).astype("float32")
logging.info("Targets subsampled")

In [8]:
# Compute tendency
ds_targets = ds_targets.diff(dim="sample")
logging.info("Tendency computed")

In [ ]:
varname = "targets" if "targets" in ds_targets.variables else list(ds_targets.data_vars)[0]
seasonality = ds_targets[varname].groupby("datetime.month").mean("sample")
da_targets = ds_targets[varname].groupby('datetime.month') - seasonality
# The above changes the chunking. So we must rechunk to make the
# xr.apply_ufunc() work
da_targets = da_targets.chunk(chunks).unify_chunks().persist()

2025-08-19 13:36:08,599 - distributed.worker.memory - WARNING - gc.collect() took 1.295s. This is usually a sign that some tasks handle too many Python objects at the same time. Rechunking the work into smaller tasks might help.


In [13]:
# Cross-channel tendency covariance
# Upper triangular channel pairs 
channels = ds_targets.sizes["channels"]
pairs = [(i, j) for i in range(channels) for j in range(i, channels)]

In [14]:
# For covariance, using a custom function like below may perform better than
# using the builtin xr.cov as that does not provide enough granularity to
# provide unbiased estimate (i.e., dividing by N-1 and not N) and may
# increase memory usage.
def cov_over_time_then_spatial_avg(a, b):
    # covariance along time per (lat,lon) then spatial mean
    a0 = a - a.mean("sample")
    b0 = b - b.mean("sample")
    num = (a0 * b0).sum("sample")
    den = a.sizes["sample"] - 1
    field_cov = num / den
    return field_cov.mean(("lat", "lon"), skipna=True)

# Use xr.corr directly for correlation computation as it is more optimized
# and reduced the memory usage by reducing intermediate arrays and
# simultaneous computation of normalizing factors.
def corr_over_time_then_spatial_avg(a, b):
    field_corr = xr.corr(a, b, dim="sample")
    return field_corr.mean(("lat", "lon"), skipna=True)

In [15]:
# Choose your function based on the value of norm 
comp_func = corr_over_time_then_spatial_avg if norm else cov_over_time_then_spatial_avg

In [16]:
# Build scalar DataArray tasks (each pair returns a scalar DataArray)
tasks = []
for (i, j) in pairs:
    xi = da_targets.isel(channels=i)
    xj = da_targets.isel(channels=j)
    tasks.append(comp_func(xi, xj))

In [ ]:
# Compute in parallel
scalars = dask.compute(*tasks)